In [ ]:
import xarray as xr
import fsspec
import matplotlib.pyplot as plt

In [ ]:
def plot_timeseries(
    ref_path,
    lat,
    lon,
    variable="precipitation",
    remote_protocol="file",
    ax=None,
):
    """
    Open a combined kerchunk reference file and plot a time series of a
    variable at the grid cell nearest to the given lat/lon.

    Parameters
    ----------
    ref_path : str
        Path to the combined kerchunk reference JSON file.
    lat : float
        Latitude in degrees (-90 to 90).
    lon : float
        Longitude in degrees (-180 to 180).
    variable : str, default "precipitation"
        Name of the data variable to plot.
    remote_protocol : str, default "file"
        Protocol used to resolve the original data files referenced in the
        kerchunk JSON (e.g. "file" for local paths, "s3" for S3 URLs).
    ax : matplotlib.axes.Axes, optional
        Existing axes to plot on. If None, a new figure/axes is created.

    Returns
    -------
    ax : matplotlib.axes.Axes
        The axes containing the plot.
    da : xarray.DataArray
        The extracted 1-D time series (for further inspection/saving).
    """
    fs = fsspec.filesystem("reference", fo=ref_path, remote_protocol=remote_protocol)
    mapper = fs.get_mapper("")
    ds = xr.open_dataset(mapper, engine="zarr", consolidated=False)

    if variable not in ds.data_vars:
        raise KeyError(
            f"'{variable}' not found in dataset. Available variables: "
            f"{list(ds.data_vars)}"
        )

    da = ds[variable].sel(lat=lat, lon=lon, method="nearest")

    actual_lat = float(da["lat"])
    actual_lon = float(da["lon"])

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 4))

    da.plot.line(ax=ax, x="time", marker="o", markersize=3)
    ax.set_title(
        f"{variable} at ({actual_lat:.2f}\u00b0, {actual_lon:.2f}\u00b0)\n"
        f"(nearest to requested {lat}\u00b0, {lon}\u00b0)"
    )
    ax.set_xlabel("Time")
    units = da.attrs.get("units") or da.attrs.get("Units", "")
    ax.set_ylabel(f"{variable}" + (f" ({units})" if units else ""))
    ax.grid(True, alpha=0.3)

    return ax, da

In [ ]:
data_file='/glade/u/home/bonnland/scratch/IMERG_d7361000/GPM_3IMERGM_07.json'

ax, da = plot_timeseries(
    data_file,
    lat=39.7,
    lon=-104.9,   # e.g. Denver
)
plt.show()